# Phase Corection
### General Story:
We got the phase and magnitude of an averaged signal from Phase on the Fly, now we want to detect frequency for PLL locking

We are assuming that the register is giving us an average of the entire adc read such that:
$$tan^{-1}(\frac{I}{Q}) = tan^{-1}(\frac{\int cos(\omega t + \phi)}{\int cos(\omega t + \phi + \frac{\pi}{4})})$$

Where by using some math and trig identities you get:

$$tan^{-1}(\frac{cos(\omega t + \phi)}{cos(\omega t + \phi + \frac{\pi}{4})}) = \phi$$

considering that 
$$\omega = \frac{\Delta \phi}{\Delta t}$$

we need two values of $\phi$ and the starting or stopping points of our read

In [1]:
#this block is for running a command on the RFsoC assuming you are 
#running one of the jupyter notebooks on the device but from a separate computer/visual studio code:
import os
dir = os.getcwd()
print(dir)
print(os.listdir())
# print(os.getcwd())
# os.system("pip install gitpython")

/home/xilinx/jupyter_notebooks
['qick', 'user_implementation', 'getting_started', 'common', 'amo_qick', '.ipynb_checkpoints', 'Welcome to Pynq.ipynb']


In [2]:
#this will pull from the github to make sure you have the most current updates

import git  # pip install gitpython
print(os.listdir())
if 'amo_qick' in os.listdir():
        print("getting into fork")
        os.chdir('amo_qick')
dir = os.getcwd()
print(dir)
g = git.cmd.Git(dir)
g.pull()

['qick', 'user_implementation', 'getting_started', 'common', 'amo_qick', '.ipynb_checkpoints', 'Welcome to Pynq.ipynb']
getting into fork
/home/xilinx/jupyter_notebooks/amo_qick


GitCommandError: Cmd('git') failed due to: exit code(1)
  cmdline: git pull
  stderr: 'fatal: unable to access 'https://github.com/davmic09/amo-qick.git/': Could not resolve host: github.com'

In [ ]:
from qick import *
from qick.asm_v2 import *
%matplotlib inline
import matplotlib.pyplot as plt

#This line is to sync to an external clock which needs to be 10 Mhz
#this line currently loads tproc1 firmware and is just to make sure everything is loaded properly
#soc = QickSoc(external_clk=True)

dir = os.getcwd()
print(dir)
#this line downloads the new hardware (just make sure thats what you want)
#soc = QickSoc()
#soc = QickSoc(bitfile =f'{dir}/tests/d_1.bit', download=True)
#as of Jan 2026: 
#rfsoc_board = most recently published bit, 
# ADC_0 = Self built file that connects adc port 0 to tproc 1
# d_1 = self built that puts a cordic loop on adc port 0 for phase analysis (doesnt work rn)
soc = QickSoc(bitfile =f'{dir}/tests/rf_board_firmware/d_1.bit', download=True)
soccfg = soc
print(soccfg)


In [ ]:
def phase_change(freq_range=(100,110), freq_division = 10, num_reps = 10):
    class ReadProgram(AveragerProgramV2):
        def _initialize(self, cfg):
            ro_ch = cfg['ro_ch']
            gen_ch = cfg['gen_ch']
            
            self.declare_gen(ch=gen_ch, nqz=1)
            self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

            self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch, 
                        style="const", 
                        freq=cfg['freq'], 
                        length=cfg['pulse_len'],
                        phase=cfg['pulse_phase'],
                        gain=cfg['gain'], 
                        )

            self.add_readoutconfig(ch=ro_ch, name="myro", freq=550, gen_ch=gen_ch, phase=cfg['ro_phase'])
            # send the config to the dynamic RO
            self.send_readoutconfig(ch=ro_ch, name="myro", t=0)
            
        def _body(self, cfg):
            self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)
            self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])
            self.wait_auto(cfg['read_wait'])
            self.read_input(ro_ch=cfg['ro_ch'])
            self.write_dmem(addr=0, src='s_port_l')
            self.write_dmem(addr=1, src='s_port_h')

    

    # prog = ReadProgram(soccfg, reps=100, final_delay=0.5, cfg=config)
    # iq_list = prog.acquire(soc, rounds=1, progress=False)
    # phase_offset = np.angle(iq_list[0][0].dot([1,1j]), deg=True)
    # config['ro_phase'] = -phase_offset
    plt.figure()
    freq_span = np.linspace(freq_range[0], freq_range[1], freq_division)
    phasebyfreq = []
    for i, freq in enumerate(freq_span):
        config = {'gen_ch': 0,
            'ro_ch': 10,
            'freq': freq,
            'trig_time': 0.40,
            'read_wait': 0.1,
            'ro_len': 0.1,
            'pulse_len': 0.025,
            'pulse_phase':0,
            'ro_phase': 0,
            'gain': 1.0
            }
        freq_data = []
        for rep in range(num_reps):
            prog = ReadProgram(soccfg, reps=1, final_delay=0.5, cfg=config)

            iq_list = prog.acquire_decimated(soc, progress = False, rounds=1)
            t = prog.get_time_axis(ro_index=0)

            # plt.plot(t, iq_list[0][:,0], label="I value")
            # plt.plot(t, iq_list[0][:,1], label="Q value")
            # plt.plot(t, np.abs(iq_list[0].dot([1,1j])), label="mag")
            # plt.legend()
            # plt.ylabel("a.u.")
            # plt.xlabel("us");

            # phase_offset = np.angle(iq_list[0].sum(axis=0).dot([1,1j]), deg=True)
            # print("phase offset:", phase_offset)

            #print("buffered readout:", iq_list[0].sum(axis=0))
            #print("feedback readout:", soc.read_mem(2,'dmem'))
            phase_maybe = soc.read_mem(2,'dmem')
            
            # print( (int(phase_maybe[0])-int(phase_maybe[1]))/(2**32-1) * 2*np.pi)
            phase_dif = int(phase_maybe[0])/(2**32-1)
            mag = int(phase_maybe[1])/(2**32-1)

            freq_data.append((phase_dif, mag))
            
        phasebyfreq.append(freq_data)
    return phasebyfreq
freq_range = (109,111)
freq_division = 21
phasebyfreq = phase_change(freq_range=freq_range, freq_division= 21, num_reps=300)